In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp /content/drive/MyDrive/caltech-101.zip .
!unzip -q caltech-101.zip

In [4]:
import shutil, os
bg = "caltech-101/BACKGROUND_Google"
if os.path.exists(bg):
    shutil.rmtree(bg)

In [5]:
import numpy as np
from torchvision import datasets

dataset = datasets.ImageFolder("caltech-101")
targets = np.array([y for _, y in dataset.samples])
num_classes = len(dataset.classes)
print("Classes:", num_classes)

rng = np.random.default_rng(42)

train_idx, val_idx, test_idx = [], [], []

for c in range(num_classes):
    idx = np.where(targets == c)[0]
    rng.shuffle(idx)
    n = len(idx)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    train_idx.extend(idx[:n_train])
    val_idx.extend(idx[n_train:n_train+n_val])
    test_idx.extend(idx[n_train+n_val:])

print(len(train_idx), len(val_idx), len(test_idx), "total:", len(dataset))

Classes: 101
6026 1256 1395 total: 8677


In [6]:
import os
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

# hyperparams
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_base = datasets.ImageFolder("caltech-101", transform=train_tf)
eval_base  = datasets.ImageFolder("caltech-101", transform=eval_tf)

class_names = train_base.classes
num_classes = len(class_names)
print("num_classes:", num_classes)

train_ds = Subset(train_base, train_idx)
val_ds   = Subset(eval_base,  val_idx)
test_ds  = Subset(eval_base,  test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| cuda:", torch.cuda.is_available())
train_ds = Subset(train_base, train_idx)
val_ds   = Subset(eval_base,  val_idx)
test_ds  = Subset(eval_base,  test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

num_classes = len(train_base.classes)
class_names = train_base.classes

num_classes: 101
device: cuda | cuda: True


In [7]:
import torch.nn as nn
from torchvision import models

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

print(model.__class__.__name__)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 60.9MB/s]


EfficientNet


In [8]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def eval_acc(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

In [9]:
from tqdm import tqdm

EPOCHS = 5
train_losses = []
val_losses = []
val_accs = []

def eval_metrics(loader):
    model.eval()
    correct, total = 0, 0
    running_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            running_loss += loss.item()

            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    avg_loss = running_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running += loss.item()

    train_loss = running / len(train_loader)
    val_loss, val_acc = eval_metrics(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

Epoch 1/5: 100%|██████████| 189/189 [00:16<00:00, 11.48it/s]


Epoch 1: train_loss=3.1012, val_loss=1.9675, val_acc=0.6871


Epoch 2/5: 100%|██████████| 189/189 [00:14<00:00, 12.95it/s]


Epoch 2: train_loss=1.4091, val_loss=0.7543, val_acc=0.8543


Epoch 3/5: 100%|██████████| 189/189 [00:15<00:00, 12.54it/s]


Epoch 3: train_loss=0.6044, val_loss=0.4057, val_acc=0.9164


Epoch 4/5: 100%|██████████| 189/189 [00:14<00:00, 12.76it/s]


Epoch 4: train_loss=0.3081, val_loss=0.2956, val_acc=0.9299


Epoch 5/5: 100%|██████████| 189/189 [00:14<00:00, 12.75it/s]


Epoch 5: train_loss=0.1841, val_loss=0.2494, val_acc=0.9323


In [11]:
import numpy as np
from sklearn.metrics import accuracy_score

os.makedirs("outputs", exist_ok=True)

model.eval()
all_y, all_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.append(pred)
        all_y.append(y.numpy())

y_true = np.concatenate(all_y)
y_pred = np.concatenate(all_pred)

test_acc = accuracy_score(y_true, y_pred)
print("Test accuracy:", test_acc)

Test accuracy: 0.9448028673835125
